# Pix2Pix Final Project - Experiments Notebook

## Goal

This notebook contains additional experiments and comparisons for the final project.

The main training notebook already trained:

1. Naive Encoder-Decoder trained with MSE loss
2. Pix2Pix with Small PatchGAN (n_layers = 3, λ = 100)
3. Pix2Pix with Large PatchGAN (n_layers = 4, λ = 100)

In this notebook, we:

- Load the trained models using **gdown**
- Evaluate the models on the test set
- Compare visual outputs
- Run additional experiments
- Create report-quality graphs
- Select the best model using both quantitative and visual analysis

The goal is not only to find the lowest numerical loss, but also to understand which model produces the best flower colorization results visually.

In [ ]:
%%capture
!pip install ipython-autotime
%load_ext autotime

In [ ]:
%%capture
!pip install -q tensorflow-datasets
!pip install -q gdown

In [ ]:
import os
import cv2
import gdown
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
from IPython.display import display

##Parameters and Folders

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = 128
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
BASE_DIR = Path("/content/pix2pix_experiments")
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "report_figures"
EXPERIMENT_MODEL_DIR = MODEL_DIR / "experiment_models"

PIX2PIX_DIR = EXPERIMENT_MODEL_DIR / "pix2pix_experiments"
METADATA_DIR = OUTPUT_DIR / "metadata"

for folder in [
    BASE_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    FIGURES_DIR,
    EXPERIMENT_MODEL_DIR,
    PIX2PIX_DIR,
    METADATA_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

##Report Plot Style

In [ ]:
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
})

def save_figure(filename):
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)


def plot_lines(series_dict, title, xlabel, ylabel, filename=None):
    plt.figure(figsize=(8, 5))

    for label, values in series_dict.items():
        plt.plot(values, label=label)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)

    if filename:
        save_figure(filename)

    plt.show()


def plot_bar_blue(df, metric, title, filename=None, ascending=None):
    plot_df = df.copy()

    if ascending is not None:
        plot_df = plot_df.sort_values(metric, ascending=ascending)

    plt.figure(figsize=(8, 5))

    plt.bar(plot_df["Model"], plot_df[metric], color="blue")

    plt.title(title)
    plt.ylabel(metric)
    plt.xticks(rotation=25, ha="right")
    plt.grid(True)

    if filename:
        save_figure(filename)

    plt.show()


def plot_tradeoff_clean(df, x_metric="L1 / MAE", y_metric="SSIM", filename=None):
    plt.figure(figsize=(8, 5))

    plt.scatter(df[x_metric], df[y_metric], color="blue")

    for i, txt in enumerate(df["Model"]):
        plt.annotate(
            txt,
            (df[x_metric].iloc[i], df[y_metric].iloc[i]),
            fontsize=8
        )

    plt.title("Model Tradeoff")
    plt.xlabel(x_metric)
    plt.ylabel(y_metric)
    plt.grid(True)

    if filename:
        save_figure(filename)

    plt.show()

## Load Main Trained Models

The main training notebook produced three trained models:

1. Naive Encoder-Decoder trained with MSE loss
2. Pix2Pix Small PatchGAN (`n_layers = 3`, `λ = 100`)
3. Pix2Pix Large PatchGAN (`n_layers = 4`, `λ = 100`)

We download them using `gdown` so this notebook can run independently without mounting Google Drive.

In [ ]:
MODEL_FILE_IDS = {
    "naive": "1r2SzBfOeqIlEX6jmqqpgFw29imjzNzmG",
    "pix2pix_small": "1S2-0l2kgPqC2FHkBGVz5wpcBXLLzcMAH",
    "pix2pix_large": "12sKEe7GX2bvs1aSRiQIluG9rl8kuA8FQ",
}

MODEL_PATHS = {
    "naive": MODEL_DIR / "best_naive_model.keras",
    "pix2pix_small": MODEL_DIR / "best_pix2pix_small_generator.keras",
    "pix2pix_large": MODEL_DIR / "best_pix2pix_large_generator.keras",
}

for model_name, file_id in MODEL_FILE_IDS.items():
    url = f"https://drive.google.com/uc?id={file_id}"
    output_path = MODEL_PATHS[model_name]

    print(f"Downloading {model_name}...")
    gdown.download(url, str(output_path), quiet=False)

print("All models downloaded successfully.")

##Load Models

In [ ]:
loaded_models = {
    "Naive MSE": keras.models.load_model(MODEL_PATHS["naive"], compile=False),
    "Pix2Pix n=3 λ=100": keras.models.load_model(MODEL_PATHS["pix2pix_small"], compile=False),
    "Pix2Pix n=4 λ=100": keras.models.load_model(MODEL_PATHS["pix2pix_large"], compile=False),
}

for name, model in loaded_models.items():
    print(f"{name} loaded successfully.")

## Dataset Loading and Preprocessing

We use the same Oxford Flowers 102 dataset and the same sketch generation pipeline as in the training notebook.

Each sample is converted into:

- Input: sketch image, shape `(128, 128, 1)`
- Target: real RGB flower image, shape `(128, 128, 3)`

Both images are normalized to `[-1, 1]`, matching the `tanh` output layer of the generator.

##Load Dataset

In [ ]:
DATA_DIR = BASE_DIR / "data"

train_ds_raw = tfds.load(
    "oxford_flowers102",
    split="train",
    data_dir=str(DATA_DIR),
    shuffle_files=False
)

val_ds_raw = tfds.load(
    "oxford_flowers102",
    split="validation",
    data_dir=str(DATA_DIR),
    shuffle_files=False
)

test_ds_raw = tfds.load(
    "oxford_flowers102",
    split="test",
    data_dir=str(DATA_DIR),
    shuffle_files=False
)

print("Raw datasets loaded successfully.")

##Sketch Generation

In [ ]:
def create_pencil_sketch_from_bgr(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(gray_blur, 50, 150)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)

    gray_f = gray.astype(np.float32)
    inv = 255.0 - gray_f
    inv_blur = cv2.GaussianBlur(inv, (0, 0), 8.0)

    denom = np.maximum(255.0 - inv_blur, 1e-3)
    pencil = np.clip(gray_f * 255.0 / denom, 0, 255).astype(np.uint8)

    return pencil


def preprocess_tfds_example(example):
    image = example["image"]

    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.uint8)

    def _make_pair(img_rgb):
        img_rgb = np.array(img_rgb)
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

        sketch = create_pencil_sketch_from_bgr(img_bgr)

        sketch = sketch.astype(np.float32) / 127.5 - 1.0
        sketch = np.expand_dims(sketch, axis=-1)

        real_rgb = img_rgb.astype(np.float32) / 127.5 - 1.0

        return sketch, real_rgb

    sketch, real = tf.numpy_function(
        _make_pair,
        [image],
        [tf.float32, tf.float32]
    )

    sketch.set_shape((IMG_SIZE, IMG_SIZE, 1))
    real.set_shape((IMG_SIZE, IMG_SIZE, 3))

    return sketch, real

In [ ]:
def denormalize(img):
    img = (img + 1.0) / 2.0
    return np.clip(img, 0.0, 1.0)

##Build Dataset Pipelines

In [ ]:
test_ds = (
    test_ds_raw
    .map(preprocess_tfds_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

train_ds_small = (
    train_ds_raw
    .take(256)
    .map(preprocess_tfds_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds_small = (
    val_ds_raw
    .take(128)
    .map(preprocess_tfds_example, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Test dataset:", test_ds)
print("Small train dataset:", train_ds_small)
print("Small validation dataset:", val_ds_small)

##Sanity Check

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))

print("Sketch batch shape:", sample_sketches.shape)
print("Target batch shape:", sample_targets.shape)

print("Sketch range:", tf.reduce_min(sample_sketches).numpy(), "to", tf.reduce_max(sample_sketches).numpy())
print("Target range:", tf.reduce_min(sample_targets).numpy(), "to", tf.reduce_max(sample_targets).numpy())

##Visualize Dataset Samples

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))

plt.figure(figsize=(10, 8))

for i in range(4):
    plt.subplot(4, 2, 2 * i + 1)
    plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
    plt.title("Input Sketch")
    plt.axis("off")

    plt.subplot(4, 2, 2 * i + 2)
    plt.imshow(denormalize(sample_targets[i]))
    plt.title("Target Image")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Visual Comparison of Main Trained Models

We first compare the three main models trained in the training notebook:

1. Naive Encoder-Decoder trained with MSE
2. Pix2Pix PatchGAN with **n_layers = 3, λ = 100**
3. Pix2Pix PatchGAN with **n_layers = 4, λ = 100**

All models receive the same sketch inputs from the test set.

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))

num_images = 4
main_predictions = {}

for model_name, model in loaded_models.items():
    main_predictions[model_name] = model.predict(sample_sketches[:num_images], verbose=0)

plt.figure(figsize=(18, 4 * num_images))

for i in range(num_images):
    plt.subplot(num_images, 5, 5 * i + 1)
    plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
    plt.title("Input Sketch")
    plt.axis("off")

    col = 2
    for model_name, preds in main_predictions.items():
        plt.subplot(num_images, 5, 5 * i + col)
        plt.imshow(denormalize(preds[i]))
        plt.title(model_name)
        plt.axis("off")
        col += 1

    plt.subplot(num_images, 5, 5 * i + 5)
    plt.imshow(denormalize(sample_targets[i]))
    plt.title("Target")
    plt.axis("off")

plt.tight_layout()
save_figure("main_models_visual_comparison.png")
plt.show()

# Quantitative Evaluation

We evaluate every model using several image reconstruction metrics:

- **L1 / MAE**: average absolute pixel difference. Lower is better.
- **MSE**: average squared pixel difference. Lower is better.
- **PSNR**: peak signal-to-noise ratio. Higher is better.
- **SSIM**: structural similarity. Higher is better.

These metrics are useful, but they do not fully capture visual realism. Since this project is an image generation task, visual comparison remains essential.

In [ ]:
def evaluate_model_on_dataset(model, dataset):
    l1_values = []
    mse_values = []
    psnr_values = []
    ssim_values = []

    for sketches, targets in dataset:
        preds = model.predict(sketches, verbose=0)

        targets_np = targets.numpy()
        preds_np = preds

        l1 = np.mean(np.abs(targets_np - preds_np))
        mse = np.mean((targets_np - preds_np) ** 2)

        targets_01 = (targets + 1.0) / 2.0
        preds_01 = (tf.convert_to_tensor(preds, dtype=tf.float32) + 1.0) / 2.0

        psnr = tf.image.psnr(targets_01, preds_01, max_val=1.0)
        ssim = tf.image.ssim(targets_01, preds_01, max_val=1.0)

        l1_values.append(l1)
        mse_values.append(mse)
        psnr_values.extend(psnr.numpy().tolist())
        ssim_values.extend(ssim.numpy().tolist())

    return {
        "L1 / MAE": float(np.mean(l1_values)),
        "MSE": float(np.mean(mse_values)),
        "PSNR": float(np.mean(psnr_values)),
        "SSIM": float(np.mean(ssim_values)),
    }

In [ ]:
evaluation_results = []

for model_name, model in loaded_models.items():
    print(f"Evaluating {model_name}...")
    metrics = evaluate_model_on_dataset(model, test_ds)

    evaluation_results.append({
        "Model": model_name,
        "Experiment Type": "Main trained model",
        **metrics
    })

main_comparison_df = pd.DataFrame(evaluation_results)
display(main_comparison_df)

##Report Graphs for Main Models

In [ ]:
plot_bar_blue(
    main_comparison_df,
    metric="L1 / MAE",
    title="Main Models Comparison - L1 / MAE",
    filename="main_models_l1_mae.png",
    ascending=True
)

In [ ]:
plot_bar_blue(
    main_comparison_df,
    metric="MSE",
    title="Main Models Comparison - MSE",
    filename="main_models_mse.png",
    ascending=True
)

In [ ]:
plot_bar_blue(
    main_comparison_df,
    metric="PSNR",
    title="Main Models Comparison - PSNR",
    filename="main_models_psnr.png",
    ascending=False
)

In [ ]:
plot_bar_blue(
    main_comparison_df,
    metric="SSIM",
    title="Main Models Comparison - SSIM",
    filename="main_models_ssim.png",
    ascending=False
)

## Main Models Discussion

The naive model is useful as a baseline because it shows what can be achieved using only pixel-level reconstruction loss.

The Pix2Pix models add an adversarial discriminator, which encourages the generator to produce more realistic textures and sharper visual details.

The comparison between `n_layers = 3` and `n_layers = 4` helps evaluate the effect of PatchGAN receptive field size.

A smaller PatchGAN focuses more on local details, while a larger PatchGAN observes a wider region and may improve global consistency.

# Naive Model Loss Experiments

The main naive model was trained with MSE loss.

In this section, we train two additional naive models with the same encoder-decoder architecture:

1. Naive Encoder-Decoder with MAE loss
2. Naive Encoder-Decoder with Combined Loss: **0.8 * MAE + 0.2 * MSE**

## Why compare these losses?

MSE gives stronger penalty to large pixel errors and often produces smoother outputs.

MAE treats pixel errors more uniformly and may preserve edges better.

The combined loss tries to balance both behaviors.

In [ ]:
def combined_loss(y_true, y_pred):
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    return 0.8 * mae + 0.2 * mse

In [ ]:
def build_naive_encoder_decoder(img_size=128, model_name="naive_encoder_decoder"):
    inputs = keras.Input(shape=(img_size, img_size, 1), name="input_sketch")

    x = layers.Conv2D(64, 4, strides=2, padding="same", activation="relu", name="enc_conv1")(inputs)

    x = layers.Conv2D(128, 4, strides=2, padding="same", name="enc_conv2")(x)
    x = layers.BatchNormalization(name="enc_bn2")(x)
    x = layers.ReLU(name="enc_relu2")(x)

    x = layers.Conv2D(256, 4, strides=2, padding="same", name="enc_conv3")(x)
    x = layers.BatchNormalization(name="enc_bn3")(x)
    x = layers.ReLU(name="enc_relu3")(x)

    x = layers.Conv2D(512, 4, strides=2, padding="same", name="bottleneck_conv")(x)
    x = layers.BatchNormalization(name="bottleneck_bn")(x)
    x = layers.ReLU(name="bottleneck_relu")(x)

    x = layers.Conv2DTranspose(256, 4, strides=2, padding="same", name="dec_deconv1")(x)
    x = layers.BatchNormalization(name="dec_bn1")(x)
    x = layers.ReLU(name="dec_relu1")(x)

    x = layers.Conv2DTranspose(128, 4, strides=2, padding="same", name="dec_deconv2")(x)
    x = layers.BatchNormalization(name="dec_bn2")(x)
    x = layers.ReLU(name="dec_relu2")(x)

    x = layers.Conv2DTranspose(64, 4, strides=2, padding="same", name="dec_deconv3")(x)
    x = layers.BatchNormalization(name="dec_bn3")(x)
    x = layers.ReLU(name="dec_relu3")(x)

    outputs = layers.Conv2DTranspose(
        3,
        4,
        strides=2,
        padding="same",
        activation="tanh",
        name="output_rgb"
    )(x)

    return keras.Model(inputs, outputs, name=model_name)

##MAE

In [ ]:
naive_mae_model = build_naive_encoder_decoder(
    IMG_SIZE,
    model_name="naive_encoder_decoder_mae"
)

naive_mae_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    loss="mae"
)

naive_callbacks_mae = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=35,
        restore_best_weights=True
    )
]


In [ ]:
mae_history = naive_mae_model.fit(
    train_ds_small,
    validation_data=val_ds_small,
    epochs=50,
    callbacks=naive_callbacks_mae,
    verbose=1
)

In [ ]:
naive_mae_model.save(EXPERIMENT_MODEL_DIR / "naive_mae_experiment.keras")

##Combined model

In [ ]:
naive_combined_model = build_naive_encoder_decoder(
    IMG_SIZE,
    model_name="naive_encoder_decoder_combined"
)

naive_combined_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-4),
    loss=combined_loss
)

naive_callbacks_combined = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=35,
        restore_best_weights=True
    )
]

In [ ]:
combined_history = naive_combined_model.fit(
    train_ds_small,
    validation_data=val_ds_small,
    epochs=50,
    callbacks=naive_callbacks_combined,
    verbose=1
)

In [ ]:
naive_combined_model.save(EXPERIMENT_MODEL_DIR / "naive_combined_experiment.keras")

##Naive Loss Curves

In [ ]:
plot_bar_blue(
    main_comparison_df,
    metric="PSNR",
    title="Main Models Comparison - PSNR",
    filename="main_models_psnr.png",
    ascending=False
)

In [ ]:
plot_lines(
    {
        "MAE Train Loss": mae_history.history["loss"],
        "MAE Validation Loss": mae_history.history["val_loss"],
    },
    title="Naive MAE Model Loss",
    xlabel="Epoch",
    ylabel="MAE Loss",
    filename="naive_mae_loss_curve.png"
)

##Visual Comparison of Naive Loss Experiments

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))
num_images = 4

naive_mse_preds = loaded_models["Naive MSE"].predict(sample_sketches[:num_images], verbose=0)
naive_mae_preds = naive_mae_model.predict(sample_sketches[:num_images], verbose=0)
naive_combined_preds = naive_combined_model.predict(sample_sketches[:num_images], verbose=0)

plt.figure(figsize=(18, 4 * num_images))

for i in range(num_images):
    plt.subplot(num_images, 5, 5 * i + 1)
    plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
    plt.title("Sketch")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 2)
    plt.imshow(denormalize(naive_mse_preds[i]))
    plt.title("Naive MSE")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 3)
    plt.imshow(denormalize(naive_mae_preds[i]))
    plt.title("Naive MAE")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 4)
    plt.imshow(denormalize(naive_combined_preds[i]))
    plt.title("Naive Combined")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 5)
    plt.imshow(denormalize(sample_targets[i]))
    plt.title("Target")
    plt.axis("off")

plt.tight_layout()
save_figure("naive_loss_visual_comparison.png")
plt.show()

##Evaluate Naive Experiments

In [ ]:
for model_name, model in {
    "Naive MAE": naive_mae_model,
    "Naive Combined": naive_combined_model
}.items():
    print(f"Evaluating {model_name}...")
    metrics = evaluate_model_on_dataset(model, test_ds)

    evaluation_results.append({
        "Model": model_name,
        "Experiment Type": "Additional naive loss experiment",
        **metrics
    })

comparison_df = pd.DataFrame(evaluation_results)
display(comparison_df)

# Pix2Pix Additional Experiments

Pix2Pix uses the following generator loss:

**Generator Total Loss = GAN Loss + λ × L1 Loss**

The main training notebook used **λ = 100**.

In this notebook, we test:

1. **λ = 50**
2. **λ = 200**
3. PatchGAN with **n_layers = 2**

The training function includes Early Stopping based on validation L1 loss.

In [ ]:
loss_object = keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss


def generator_loss(disc_generated_output, gen_output, target, lambda_l1):
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
    total_gen_loss = gan_loss + lambda_l1 * l1_loss
    return total_gen_loss, gan_loss, l1_loss

##Generator and Discriminator Blocks

In [ ]:
def downsample(filters, size, apply_batchnorm=True, block_name="down"):
    initializer = tf.random_normal_initializer(0.0, 0.02)

    result = keras.Sequential(name=block_name)
    result.add(layers.Conv2D(
        filters, size, strides=2, padding="same",
        kernel_initializer=initializer,
        use_bias=False,
        name=f"{block_name}_conv"
    ))

    if apply_batchnorm:
        result.add(layers.BatchNormalization(name=f"{block_name}_bn"))

    result.add(layers.LeakyReLU(name=f"{block_name}_leaky_relu"))
    return result


def upsample(filters, size, apply_dropout=False, block_name="up"):
    initializer = tf.random_normal_initializer(0.0, 0.02)

    result = keras.Sequential(name=block_name)
    result.add(layers.Conv2DTranspose(
        filters, size, strides=2, padding="same",
        kernel_initializer=initializer,
        use_bias=False,
        name=f"{block_name}_conv_transpose"
    ))

    result.add(layers.BatchNormalization(name=f"{block_name}_bn"))

    if apply_dropout:
        result.add(layers.Dropout(0.5, name=f"{block_name}_dropout"))

    result.add(layers.ReLU(name=f"{block_name}_relu"))
    return result

##U-Net Generator

In [ ]:
def build_unet_generator(img_size=128, model_name="pix2pix_unet_generator"):
    inputs = layers.Input(
        shape=(img_size, img_size, 1),
        name="generator_input_sketch"
    )

    down_stack = [
        downsample(64, 4, apply_batchnorm=False, block_name="gen_down_1"),
        downsample(128, 4, block_name="gen_down_2"),
        downsample(256, 4, block_name="gen_down_3"),
        downsample(512, 4, block_name="gen_down_4"),
        downsample(512, 4, block_name="gen_down_5"),
        downsample(512, 4, block_name="gen_down_6"),
        downsample(512, 4, block_name="gen_bottleneck"),
    ]

    up_stack = [
        upsample(512, 4, apply_dropout=True, block_name="gen_up_1"),
        upsample(512, 4, apply_dropout=True, block_name="gen_up_2"),
        upsample(512, 4, apply_dropout=True, block_name="gen_up_3"),
        upsample(256, 4, block_name="gen_up_4"),
        upsample(128, 4, block_name="gen_up_5"),
        upsample(64, 4, block_name="gen_up_6"),
    ]

    initializer = tf.random_normal_initializer(0.0, 0.02)

    last = layers.Conv2DTranspose(
        3,
        4,
        strides=2,
        padding="same",
        kernel_initializer=initializer,
        activation="tanh",
        name="generator_output_rgb"
    )

    x = inputs
    skips = []

    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    for idx, (up, skip) in enumerate(zip(up_stack, skips), start=1):
        x = up(x)
        x = layers.Concatenate(name=f"gen_skip_connection_{idx}")([x, skip])

    outputs = last(x)

    return keras.Model(
        inputs=inputs,
        outputs=outputs,
        name=model_name
    )

##PatchGAN Discriminator

In [ ]:
def build_patch_discriminator(img_size=128, base_filters=64, n_layers=3, name="patchgan_discriminator"):
    initializer = tf.random_normal_initializer(0.0, 0.02)

    inp = layers.Input(
        shape=(img_size, img_size, 1),
        name="disc_input_sketch"
    )

    tar = layers.Input(
        shape=(img_size, img_size, 3),
        name="disc_input_target_or_generated"
    )

    x = layers.Concatenate(name="disc_concat_sketch_and_image")([inp, tar])

    filters = base_filters

    x = layers.Conv2D(
        filters,
        4,
        strides=2,
        padding="same",
        kernel_initializer=initializer,
        name="disc_conv_1"
    )(x)
    x = layers.LeakyReLU(name="disc_leaky_relu_1")(x)

    for i in range(1, n_layers):
        filters = min(base_filters * (2 ** i), 512)
        stride = 1 if i == n_layers - 1 else 2

        x = layers.Conv2D(
            filters,
            4,
            strides=stride,
            padding="same",
            kernel_initializer=initializer,
            use_bias=False,
            name=f"disc_conv_{i + 1}"
        )(x)

        x = layers.BatchNormalization(name=f"disc_batchnorm_{i + 1}")(x)
        x = layers.LeakyReLU(name=f"disc_leaky_relu_{i + 1}")(x)

    x = layers.Conv2D(
        1,
        4,
        strides=1,
        padding="same",
        kernel_initializer=initializer,
        name="disc_output_patch_map"
    )(x)

    return keras.Model(
        inputs=[inp, tar],
        outputs=x,
        name=name
    )

##Pix2Pix Fit Function with Early Stopping

In [ ]:
def save_experiment_metadata(path, metadata):
    with open(path, "w") as f:
        json.dump(metadata, f, indent=4)

In [ ]:
def fit_pix2pix(train_dataset, val_dataset, epochs, generator, discriminator, experiment_name, lambda_l1=100):
    patience = 30
    epochs_without_improvement = 0

    history = {
        "gen_total_loss": [],
        "gen_gan_loss": [],
        "gen_l1_loss": [],
        "disc_loss": [],
        "val_l1_loss": []
    }

    generator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)
    discriminator_optimizer = keras.optimizers.Adam(2e-4, beta_1=0.5)

    sample_sketches, sample_targets = next(iter(val_dataset))

    experiment_dir = PIX2PIX_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    best_generator_path = experiment_dir / "best_generator.keras"
    best_discriminator_path = experiment_dir / "best_discriminator.keras"
    last_generator_path = experiment_dir / "last_generator.keras"
    last_discriminator_path = experiment_dir / "last_discriminator.keras"
    history_csv_path = experiment_dir / "history.csv"
    metadata_path = METADATA_DIR / f"{experiment_name}.json"

    best_val_l1 = float("inf")
    best_epoch = -1

    @tf.function
    def train_step(input_image, target):
        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
            gen_output = generator(input_image, training=True)

            disc_real_output = discriminator([input_image, target], training=True)
            disc_generated_output = discriminator([input_image, gen_output], training=True)

            total_gen_loss, gan_loss, l1_loss = generator_loss(
                disc_generated_output, gen_output, target, lambda_l1
            )
            disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

        generator_gradients = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
        discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

        generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
        discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

        return total_gen_loss, gan_loss, l1_loss, disc_loss

    for epoch in range(epochs):
        gen_total_list = []
        gen_gan_list = []
        gen_l1_list = []
        disc_list = []

        for input_image, target in train_dataset:
            gen_total_loss, gan_loss, l1_loss, disc_loss = train_step(input_image, target)

            gen_total_list.append(gen_total_loss.numpy())
            gen_gan_list.append(gan_loss.numpy())
            gen_l1_list.append(l1_loss.numpy())
            disc_list.append(disc_loss.numpy())

        history["gen_total_loss"].append(np.mean(gen_total_list))
        history["gen_gan_loss"].append(np.mean(gen_gan_list))
        history["gen_l1_loss"].append(np.mean(gen_l1_list))
        history["disc_loss"].append(np.mean(disc_list))

        # Validation
        val_l1_list = []
        for val_input, val_target in val_dataset:
            val_pred = generator(val_input, training=False)
            val_l1 = tf.reduce_mean(tf.abs(val_target - val_pred))
            val_l1_list.append(val_l1.numpy())

        current_val_l1 = float(np.mean(val_l1_list))
        history["val_l1_loss"].append(current_val_l1)

        # Best model check
        if current_val_l1 < best_val_l1:
            best_val_l1 = current_val_l1
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            generator.save(best_generator_path)
            discriminator.save(best_discriminator_path)

            print(f"New best model saved at epoch {epoch + 1} with Val L1 = {best_val_l1:.4f}")
        else:
            epochs_without_improvement += 1

        # Logging
        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Gen Total: {history['gen_total_loss'][-1]:.4f} | "
            f"Gen GAN: {history['gen_gan_loss'][-1]:.4f} | "
            f"Gen L1: {history['gen_l1_loss'][-1]:.4f} | "
            f"Val L1: {history['val_l1_loss'][-1]:.4f} | "
            f"Disc: {history['disc_loss'][-1]:.4f}"
        )

        # Visualization
        preds = generator(sample_sketches[:3], training=False).numpy()

        plt.figure(figsize=(12, 12))
        for i in range(3):
            plt.subplot(3, 3, 3 * i + 1)
            plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
            plt.title("Sketch")
            plt.axis("off")

            plt.subplot(3, 3, 3 * i + 2)
            plt.imshow(denormalize(preds[i]))
            plt.title(f"Generated - Epoch {epoch + 1}")
            plt.axis("off")

            plt.subplot(3, 3, 3 * i + 3)
            plt.imshow(denormalize(sample_targets[i]))
            plt.title("Target")
            plt.axis("off")

        plt.tight_layout()
        plt.savefig(experiment_dir / f"epoch_{epoch + 1:03d}.png")
        plt.show()

        #Early Stopping
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    # Save last models
    generator.save(last_generator_path)
    discriminator.save(last_discriminator_path)

    pd.DataFrame(history).to_csv(history_csv_path, index=False)

    save_experiment_metadata(
        metadata_path,
        {
            "experiment_name": experiment_name,
            "model_type": "pix2pix",
            "lambda_l1": lambda_l1,
            "generator_optimizer": "Adam(beta_1=0.5, lr=2e-4)",
            "discriminator_optimizer": "Adam(beta_1=0.5, lr=2e-4)",
            "epochs_requested": epochs,
            "best_val_l1": best_val_l1,
            "best_epoch": best_epoch,
            "best_generator_path": str(best_generator_path),
            "best_discriminator_path": str(best_discriminator_path),
            "last_generator_path": str(last_generator_path),
            "last_discriminator_path": str(last_discriminator_path),
            "history_csv_path": str(history_csv_path)
        }
    )

    return history

##Plot Pix2Pix Training Curves

In [ ]:
def plot_pix2pix_history(history, title, filename):
    plot_lines(
        {
            "Generator Total Loss": history["gen_total_loss"],
            "Generator GAN Loss": history["gen_gan_loss"],
            "Generator L1 Loss": history["gen_l1_loss"],
            "Discriminator Loss": history["disc_loss"],
            "Validation L1 Loss": history["val_l1_loss"],
        },
        title=title,
        xlabel="Epoch",
        ylabel="Loss",
        filename=filename
    )

##Experiment 1: Pix2Pix λ = 50

In [ ]:
lambda_50_generator = build_unet_generator(
    IMG_SIZE,
    model_name="pix2pix_generator_lambda_50"
)

lambda_50_discriminator = build_patch_discriminator(
    IMG_SIZE,
    n_layers=3,
    name="patchgan_discriminator_lambda_50"
)

history_lambda_50 = fit_pix2pix(
    train_dataset=train_ds_small,
    val_dataset=val_ds_small,
    epochs=30,
    generator=lambda_50_generator,
    discriminator=lambda_50_discriminator,
    lambda_l1=50,
    experiment_name="pix2pix_lambda_50",
)



In [ ]:
plot_pix2pix_history(
    history_lambda_50,
    title="Pix2Pix λ=50 - Training Curves",
    filename="pix2pix_lambda_50_training_curves.png"
)

##Experiment 2: Pix2Pix λ = 200

In [ ]:
lambda_200_generator = build_unet_generator(
    IMG_SIZE,
    model_name="pix2pix_generator_lambda_200"
)

lambda_200_discriminator = build_patch_discriminator(
    IMG_SIZE,
    n_layers=3,
    name="patchgan_discriminator_lambda_200"
)

history_lambda_200 = fit_pix2pix(
    train_dataset=train_ds_small,
    val_dataset=val_ds_small,
    epochs=30,
    generator=lambda_200_generator,
    discriminator=lambda_200_discriminator,
    lambda_l1=200,
    experiment_name="pix2pix_lambda_200",
)


In [ ]:

plot_pix2pix_history(
    history_lambda_200,
    title="Pix2Pix λ=200 - Training Curves",
    filename="pix2pix_lambda_200_training_curves.png"
)

##Experiment 3: PatchGAN n_layers = 2

In [ ]:
patch_n2_generator = build_unet_generator(
    IMG_SIZE,
    model_name="pix2pix_generator_patch_n2"
)

patch_n2_discriminator = build_patch_discriminator(
    IMG_SIZE,
    n_layers=2,
    name="patchgan_discriminator_n2"
)

history_patch_n2 = fit_pix2pix(
    train_dataset=train_ds_small,
    val_dataset=val_ds_small,
    epochs=30,
    generator=patch_n2_generator,
    discriminator=patch_n2_discriminator,
    lambda_l1=100,
    experiment_name="pix2pix_patch_n2_lambda_100",
)



In [ ]:
plot_pix2pix_history(
    history_patch_n2,
    title="Pix2Pix PatchGAN n=2 - Training Curves",
    filename="pix2pix_patch_n2_training_curves.png"
)

## Visual Comparison of Pix2Pix Lambda Experiments

In this section, we compare the effect of changing the L1 weight λ.

- `λ = 50`: more relative influence to GAN loss
- `λ = 100`: balanced setting from the main training notebook
- `λ = 200`: more influence to reconstruction loss

Visual comparison is important because numerical losses do not fully describe image realism.

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))
num_images = 4

lambda_100_preds = loaded_models["Pix2Pix n=3 λ=100"].predict(
    sample_sketches[:num_images],
    verbose=0
)

lambda_50_preds = lambda_50_generator.predict(
    sample_sketches[:num_images],
    verbose=0
)

lambda_200_preds = lambda_200_generator.predict(
    sample_sketches[:num_images],
    verbose=0
)

plt.figure(figsize=(18, 4 * num_images))

for i in range(num_images):
    plt.subplot(num_images, 5, 5 * i + 1)
    plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
    plt.title("Sketch")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 2)
    plt.imshow(denormalize(lambda_50_preds[i]))
    plt.title("Pix2Pix λ=50")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 3)
    plt.imshow(denormalize(lambda_100_preds[i]))
    plt.title("Pix2Pix λ=100")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 4)
    plt.imshow(denormalize(lambda_200_preds[i]))
    plt.title("Pix2Pix λ=200")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 5)
    plt.imshow(denormalize(sample_targets[i]))
    plt.title("Target")
    plt.axis("off")

plt.tight_layout()
save_figure("pix2pix_lambda_visual_comparison.png")
plt.show()

## Visual Comparison of PatchGAN Receptive Field

We compare three PatchGAN configurations:

1. `n_layers = 2`
2. `n_layers = 3`
3. `n_layers = 4`

Increasing `n_layers` increases the effective receptive field of the discriminator.

A smaller discriminator focuses more on local details.  
A larger discriminator considers a wider region and may improve global consistency.

In [ ]:
sample_sketches, sample_targets = next(iter(test_ds))
num_images = 4

patch_n2_preds = patch_n2_generator.predict(
    sample_sketches[:num_images],
    verbose=0
)

patch_n3_preds = loaded_models["Pix2Pix n=3 λ=100"].predict(
    sample_sketches[:num_images],
    verbose=0
)

patch_n4_preds = loaded_models["Pix2Pix n=4 λ=100"].predict(
    sample_sketches[:num_images],
    verbose=0
)

plt.figure(figsize=(18, 4 * num_images))

for i in range(num_images):
    plt.subplot(num_images, 5, 5 * i + 1)
    plt.imshow(denormalize(sample_sketches[i, :, :, 0]), cmap="gray")
    plt.title("Sketch")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 2)
    plt.imshow(denormalize(patch_n2_preds[i]))
    plt.title("PatchGAN n=2")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 3)
    plt.imshow(denormalize(patch_n3_preds[i]))
    plt.title("PatchGAN n=3")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 4)
    plt.imshow(denormalize(patch_n4_preds[i]))
    plt.title("PatchGAN n=4")
    plt.axis("off")

    plt.subplot(num_images, 5, 5 * i + 5)
    plt.imshow(denormalize(sample_targets[i]))
    plt.title("Target")
    plt.axis("off")

plt.tight_layout()
save_figure("pix2pix_patchgan_receptive_field_visual_comparison.png")
plt.show()

##Evaluate All Pix2Pix Experiments

# Final Model Comparison

In this section, we compare all models together:

- Naive models
- Pix2Pix main models
- Additional experiments

This allows us to analyze both:

- Quantitative performance (metrics)
- Qualitative performance (visual results)

The goal is to select the best model based on both criteria.

In [ ]:
plot_bar_blue(
    comparison_df,
    metric="L1 / MAE",
    title="Final Comparison - L1 / MAE",
    filename="final_l1_mae.png",
    ascending=True
)

In [ ]:
plot_bar_blue(
    comparison_df,
    metric="MSE",
    title="Final Comparison - MSE",
    filename="final_mse.png",
    ascending=True
)

In [ ]:
plot_bar_blue(
    comparison_df,
    metric="PSNR",
    title="Final Comparison - PSNR",
    filename="final_psnr.png",
    ascending=False
)

In [ ]:
plot_bar_blue(
    comparison_df,
    metric="SSIM",
    title="Final Comparison - SSIM",
    filename="final_ssim.png",
    ascending=False
)

##Tradeoff Graph

In [ ]:
plot_tradeoff_clean(
    comparison_df,
    x_metric="L1 / MAE",
    y_metric="SSIM",
    filename="final_tradeoff_l1_ssim.png"
)

# Final Conclusions

From the experiments, we can conclude the following:

1. The naive model provides a useful baseline, but it is limited because it relies only on pixel-wise reconstruction loss.

2. Using MAE instead of MSE slightly improves structure preservation, but the outputs remain blurry.

3. The combined loss (MAE + MSE) balances both behaviors, but still lacks realistic texture.

4. Pix2Pix significantly improves image quality by combining:
   - Adversarial loss (realism)
   - L1 loss (structure preservation)

5. The PatchGAN discriminator plays an important role:
   - Smaller receptive field (n=2, n=3) focuses on local texture
   - Larger receptive field (n=4) improves global consistency

6. The λ parameter controls the balance between realism and reconstruction:
   - Lower λ (50): more realistic but less accurate
   - Higher λ (200): more accurate but smoother
   - λ=100 provides a good balance

7. Quantitative metrics such as L1, MSE, PSNR, and SSIM are useful,
   but they do not fully capture visual realism.

8. Visual comparison shows that Pix2Pix with PatchGAN n=3 produces
   the sharpest and most realistic flower images.

---

## Final Model Selection

The selected model is:

**Pix2Pix with PatchGAN (n_layers = 3) and λ = 100**

This model achieves the best balance between:

- Sharpness
- Color realism
- Structural consistency

It will be used in the test environment notebook.

In [ ]:
final_csv_path = OUTPUT_DIR / "final_experiments_results.csv"

comparison_df.to_csv(final_csv_path, index=False)

print("Saved final comparison table to:", final_csv_path)